# DataPulse — Natural-Language → SQL Agent

Ask a question in plain English; watch a **tool-calling agent** (Groq, Llama-3.3-70B) plan the
query, run it, and answer. Each turn is an explicit tool call, so the reasoning is a **readable
trace** rather than one opaque generation:

1. **`get_schema_context`** — pull the relevant tables, **join paths**, and **canonical metric
   definitions** from a Neo4j *metadata* knowledge graph.
2. **`sample_values`** *(optional)* — inspect a column's distinct values to resolve a
   categorical filter instead of guessing.
3. **`run_sql`** — execute a **read-only** query against SQLite and read the rows.
4. Write the final natural-language answer.

This is where the knowledge graph earns its keep: the model *discovers* the schema, the joins,
and the business metrics by calling tools.

## Prerequisites

- `.env` with `GROQ_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` (see `.env.sample`).
- `src/db/sales.db` present (git-tracked, or run `/run-pipeline`).
- The Neo4j **metadata** knowledge graph built and fresh in Aura:
  `uv run python -m src.knowledge_graph.builder` (or `/run-pipeline`).

The outputs below are from a **real run**. The live cells degrade gracefully when creds are
missing, so the notebook still reads end-to-end without any keys.

## Setup — load credentials & check the environment

In [1]:
import json
import logging
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


def _find_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "pyproject.toml").exists():
            return cand
    return start.resolve()


PROJECT_ROOT = _find_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

from src.db.loader import DB_PATH

_KEYS = ["GROQ_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]
HAVE_CREDS = all(os.getenv(k) for k in _KEYS)
for k in _KEYS:
    print(f"{k:16}: {'set' if os.getenv(k) else 'MISSING'}")
print(f"sqlite db       : {DB_PATH.name} ({'present' if DB_PATH.exists() else 'MISSING - run /run-pipeline'})")
print(f"live agent      : {'enabled' if HAVE_CREDS else 'disabled (set creds in .env to run the live cells)'}")

GROQ_API_KEY    : set
NEO4J_URI       : set
NEO4J_USERNAME  : set
NEO4J_PASSWORD  : set
sqlite db       : sales.db (present)
live agent      : enabled


## 1. Ask a Question — the live agent

`answer_question` wires the real Groq client and the knowledge-graph-backed tools, then runs the
loop. The `ask` helper below prints the tool-call trace, the SQL the model settled on, and the
final answer — and skips cleanly if creds are missing.

> **Watch the trap:** "revenue" must come from `order_items.line_total`, not `invoices.amount`.
> The metric glossary in the knowledge graph is what keeps the model honest.

In [2]:
from text2sql.agent.agent import RateLimitExhausted, answer_question


def ask(question: str):
    print(f"Q: {question}\n")
    if not HAVE_CREDS:
        print("[live agent disabled - set GROQ/NEO4J creds in .env to run this]")
        return None
    try:
        res = answer_question(
            question,
            groq_key=os.getenv("GROQ_API_KEY"),
            uri=os.getenv("NEO4J_URI"),
            user=os.getenv("NEO4J_USERNAME"),
            password=os.getenv("NEO4J_PASSWORD"),
            db_path=DB_PATH,
        )
    except RateLimitExhausted as exc:
        print(f"[Groq rate limit - try again later: {exc}]")
        return None
    for step in res.trace:
        if step.kind == "tool":
            print(f"  -> {step.tool}: {step.observation}")
    print(f"\nSQL: {res.last_sql}")
    print(f"A  : {res.answer}")
    return res


_ = ask("What is the total sales revenue?")

Q: What is the total sales revenue?



  -> get_schema_context: tables=['campaign_metrics', 'campaigns', 'invoices', 'marketing_budgets', 'order_items', 'orders', 'products', 'returns', 'sales_reps', 'sales_targets'], metrics=['total revenue', 'average order value', 'gross margin', 'target attainment rate']
  -> run_sql: 1 rows, columns=['SUM(line_total)'], first=[[7782964.89]]

SQL: SELECT SUM(line_total) FROM order_items
A  : The total sales revenue is $7,782,964.89.


## 2. Why the Knowledge Graph Matters — a fan-out trap

Average order value must divide by `COUNT(DISTINCT order_id)`; dividing by `COUNT(*)` of line
items fans out and understates it. The graph carries both the canonical metric expression and
the join **cardinality hints**, so the model steers around the trap.

In [3]:
_ = ask("What is the average order value?")

Q: What is the average order value?



  -> get_schema_context: tables=['invoices', 'order_items', 'orders', 'products', 'returns', 'sales_reps', 'sales_targets'], metrics=['total revenue', 'average order value', 'gross margin', 'target attainment rate']
  -> run_sql: 1 rows, columns=['average_order_value'], first=[[3891.4824449999996]]

SQL: SELECT SUM(line_total) / COUNT(DISTINCT order_id) AS average_order_value FROM order_items;
A  : The average order value is $3891.48.


## 3. How the Loop Works — and how it's tested without a live model

The loop, `run_agent`, is **dependency-injected** over an `llm_fn` and a `tool_fns` registry, so
it can be driven by a scripted fake model and the *real* tools — no Groq or Neo4j needed. That is
exactly how the unit tests exercise it. Below, a deterministic `scripted_llm` plans one query;
`run_agent` feeds the **real** SQLite result back, and the fake writes the answer from those rows.

In [4]:
from text2sql.agent.agent import TOOL_SCHEMAS, LLMResponse, ToolCall, run_agent
from text2sql.agent.tools import run_sql

DEMO_SQL = (
    "SELECT c.category_name, ROUND(SUM(oi.line_total), 2) AS revenue "
    "FROM order_items oi "
    "JOIN products   p ON oi.product_id = p.product_id "
    "JOIN categories c ON p.category_id = c.category_id "
    "GROUP BY c.category_name ORDER BY revenue DESC LIMIT 3"
)


def scripted_llm(messages, tool_schemas):
    """A deterministic stand-in for Groq. Turn 1: ask to run one query. Turn 2
    (after run_agent feeds the real tool result back in): write the answer."""
    tool_results = [m for m in messages if m["role"] == "tool"]
    if not tool_results:
        return LLMResponse(content=None, tool_calls=[ToolCall("call-1", "run_sql", {"sql": DEMO_SQL})])
    rows = json.loads(tool_results[-1]["content"]).get("rows", [])
    ranking = ", ".join(f"{name} (${rev:,.0f})" for name, rev in rows)
    return LLMResponse(content=f"Top 3 categories by revenue: {ranking}.")


result = run_agent(
    "What are the top 3 product categories by revenue?",
    llm_fn=scripted_llm,
    tool_fns={"run_sql": lambda sql: run_sql(sql, DB_PATH)},  # the real tool
    tool_schemas=TOOL_SCHEMAS,
)
for step in result.trace:
    print(f"  {step.kind:6} {step.tool or '':16} {step.observation}")
print(f"\nanswer  : {result.answer}")
print(f"stopped : {result.stopped}")

  tool   run_sql          3 rows, columns=['category_name', 'revenue'], first=[["Women's Clothing", 1093955.64]]
  final                   Top 3 categories by revenue: Women's Clothing ($1,093,956), Clothing ($1,035,810), Smartphones ($981,425).

answer  : Top 3 categories by revenue: Women's Clothing ($1,093,956), Clothing ($1,035,810), Smartphones ($981,425).
stopped : final


## 4. The Read-Only SQL Guard

`run_sql` is not a bare cursor. Every query is gated before it touches the database: a
`SELECT`/`WITH` allowlist, single-statement only, a forbidden-keyword block, a read-only
connection, and a statement timeout + row cap. `run_agent` also catches any tool exception, so a
bad query never crashes a run.

In [5]:
from text2sql.agent.tools import read_only_violation

candidates = [
    "SELECT COUNT(*) FROM orders",
    "WITH t AS (SELECT 1 AS n) SELECT * FROM t",
    "DROP TABLE orders",
    "UPDATE orders SET status = 'X'",
    "SELECT 1; DELETE FROM orders",
    "PRAGMA table_info(orders)",
]
for sql in candidates:
    v = read_only_violation(sql)
    verdict = "ALLOW" if v is None else f"REJECT ({v})"
    print(f"  {verdict:42} | {sql}")

  ALLOW                                      | SELECT COUNT(*) FROM orders
  ALLOW                                      | WITH t AS (SELECT 1 AS n) SELECT * FROM t
  REJECT (only SELECT / WITH read queries are allowed) | DROP TABLE orders
  REJECT (only SELECT / WITH read queries are allowed) | UPDATE orders SET status = 'X'
  REJECT (multiple statements are not allowed) | SELECT 1; DELETE FROM orders
  REJECT (only SELECT / WITH read queries are allowed) | PRAGMA table_info(orders)


## Under the Hood

- `text2sql/agent/agent.py` — the dependency-injected loop (`run_agent`) and the real Groq wiring
  (`answer_question`).
- `text2sql/agent/tools.py` — `get_schema_context`, `sample_values`, `run_sql`, and the read-only
  guard.
- `src/knowledge_graph/retriever.py` — the join-path planner over the Neo4j metadata graph.
- `docs/query_engine.md` — the design writeup. `text2sql/eval/` — the accuracy eval harness.